# TALLYMARK: B2B Identity Enrichment adapter training

This notebook builds and trains the Hybrid `B2B_IDENTITY_ENRICHMENT` adapter using the pinned Telegraph kit. The data-gate child intent is `B2B_IDENTITY_ENRICHMENT_GENERAL`. It does not sign transactions, pay a bond, register an adapter, or host a public executor.

The training corpus contains 272 generated preference pairs from 17 issuer-referenced companies. Three additional companies are held out. Pair labels are generated counterfactuals with unverified LLM provenance; workforce definitions and dates vary by issuer. The industry labels are normalized and still require independent review.

Run on a Kaggle GPU. Stop if CUDA is unavailable.

In [ ]:
import os, sys, json, pathlib, subprocess, shutil, hashlib, tarfile, urllib.request
from pathlib import Path

WORK = Path('/kaggle/working')

def run(cmd, cwd=None, check=True):
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, check=check)

run(['nvidia-smi'])
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU attached. Select Accelerator → GPU and restart the session.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
KIT = WORK / 'telegraph-wasm-baseline'
PROJECT = WORK / 'tallymark'
if not KIT.exists():
    run(['git','clone','https://github.com/telegraphprotocol/telegraph-wasm-baseline.git',str(KIT)])
run(['git','checkout','1ed063e2f0066cfaf0e124d66ec6a17b7a6121fd'],cwd=KIT)
if not PROJECT.exists():
    run(['git','clone','--branch','feat/b2b-identity-enrichment-adapter','https://github.com/onchaindc/tallymark.git',str(PROJECT)])
else:
    run(['git','fetch','origin','feat/b2b-identity-enrichment-adapter'],cwd=PROJECT)
    run(['git','checkout','feat/b2b-identity-enrichment-adapter'],cwd=PROJECT)
print('Kit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=KIT,text=True).strip())
print('TALLYMARK:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT,text=True).strip())


In [ ]:
# Install Python dependencies without replacing Kaggle's CUDA-enabled PyTorch.
run([sys.executable,'-m','pip','install','--upgrade','transformers>=4.48','peft','numpy','onnx','onnxruntime','huggingface_hub'])
import torch
assert torch.cuda.is_available(), 'CUDA disappeared after dependency installation'
print('CUDA remains available:',torch.cuda.get_device_name(0))


In [ ]:
# Build the new B2B corpus and run the pinned Telegraph data gate before training.
run([sys.executable,str(PROJECT/'training/build_b2b_preference_pairs.py')])
DATA = PROJECT/'training/b2b-identity-enrichment-pairs.jsonl'
assert DATA.exists()
run(['node','--test',str(PROJECT/'test/b2b-heldout.test.js')])
run([sys.executable,'-m','float_runtime.train_adapter','--intent','B2B_IDENTITY_ENRICHMENT_GENERAL','--data',str(DATA),'--check-data'],cwd=KIT)


In [ ]:
# Fetch the official foundation metadata and node-compatible tokenizer.
FOUNDATION = KIT/'foundation'
FOUNDATION.mkdir(exist_ok=True)
release = 'https://github.com/telegraphprotocol/telegraph-wasm-baseline/releases/download/modernbert-base-lora16-v1'
for name in ['foundation.json','SHA256SUMS','tokenizer.json']:
    run(['curl','-fL','-o',str(FOUNDATION/name),f'{release}/{name}'])
run(['sha256sum','-c','--ignore-missing','SHA256SUMS'],cwd=FOUNDATION)
spec = json.loads((FOUNDATION/'foundation.json').read_text())
TOKENIZER = FOUNDATION/'tokenizer.json'
tokenizer_hash = hashlib.sha256(TOKENIZER.read_bytes()).hexdigest()
assert tokenizer_hash == spec['tokenizer_hash'], (tokenizer_hash,spec['tokenizer_hash'])
print('Official tokenizer:',tokenizer_hash)
print('Foundation:',spec['foundation_version'],spec['foundation_hash'])


In [ ]:
# Reconstruct the node's ModernBERT base directory with the official tokenizer.
from huggingface_hub import snapshot_download
snapshot = Path(snapshot_download(spec['base_model']))
NODE_BASE = FOUNDATION/'modernbert-node-tokenizer'
NODE_BASE.mkdir(exist_ok=True)
for p in snapshot.iterdir():
    if p.name == 'tokenizer.json':
        continue
    dest = NODE_BASE/p.name
    if not dest.exists():
        dest.symlink_to(p)
shutil.copy2(TOKENIZER,NODE_BASE/'tokenizer.json')
sys.path.insert(0,str(KIT))
from transformers import AutoTokenizer
from float_runtime.train_adapter import check_tokenizer
tok = AutoTokenizer.from_pretrained(str(NODE_BASE),trust_remote_code=True)
check_tokenizer(tok)
print('Tokenizer compatibility: PASS')
print('Model weights SHA256:',hashlib.sha256((NODE_BASE/'model.safetensors').read_bytes()).hexdigest())


In [ ]:
# Train the Hybrid parent-family adapter. The pinned trainer may raise a post-save NameError
# in its final display code; continue only if a complete result.json was actually written.
OUT = KIT/'runs/B2B_IDENTITY_ENRICHMENT'
cmd = [sys.executable,'-m','float_runtime.train_adapter','--intent','B2B_IDENTITY_ENRICHMENT','--data',str(DATA),'--out',str(OUT),'--base-model',str(NODE_BASE),'--epochs','3','--max-length','512','--batch-size','1','--grad-accum','16']
proc = run(cmd,cwd=KIT,check=False)
RESULT_PATH = OUT/'result.json'
if not RESULT_PATH.exists():
    raise RuntimeError(f'Training did not save result.json (exit {proc.returncode})')
RESULT = json.loads(RESULT_PATH.read_text())
print(json.dumps({k:RESULT.get(k) for k in ['intent','train_pairs','gate_split','n_eval','pairwise_accuracy','length_baseline','beats_length_baseline','verdict','fail_reason']},indent=2))
if RESULT.get('verdict') != 'PASS':
    raise RuntimeError('Held-out model gate failed. Do not export or register this model.')


In [ ]:
# Verify actual model and tokenizer hashes before correcting the trainer's local-path metadata.
model_hash = hashlib.sha256((NODE_BASE/'model.safetensors').read_bytes()).hexdigest()
tok_hash = hashlib.sha256(TOKENIZER.read_bytes()).hexdigest()
assert model_hash == RESULT['base_model_sha256'], 'Base model weights do not match result.json'
assert tok_hash == RESULT['tokenizer_sha256'], 'Tokenizer does not match result.json'
RESULT['base_model'] = spec['base_model']
RESULT_PATH.write_text(json.dumps(RESULT,indent=2)+'\n')
(OUT/'adapter').mkdir(exist_ok=True)
shutil.copy2(TOKENIZER,OUT/'adapter/tokenizer.json')
print('Verified model hash:',model_hash)
print('Verified node tokenizer hash:',tok_hash)
print('Canonical base model metadata:',RESULT['base_model'])


In [ ]:
# Export and run the protocol's adapterbuild parser/shape/hash gate.
run([sys.executable,'-m','float_runtime.export_adapter','--run',str(OUT),'--intent','B2B_IDENTITY_ENRICHMENT','--class','hybrid','--foundation',str(FOUNDATION/'foundation.json')],cwd=KIT)
PACKAGE = OUT/'B2B_IDENTITY_ENRICHMENT.safetensors'
assert PACKAGE.exists(), 'Export did not create the safetensors package'

def go_version():
    try:
        import re
        out = subprocess.check_output(['go','version'],text=True,stderr=subprocess.STDOUT)
        m = re.search(r'go(\\d+)\\.(\\d+)',out)
        return (int(m.group(1)),int(m.group(2))) if m else (0,0)
    except Exception:
        return (0,0)

if go_version() < (1,25):
    tgz = WORK/'go1.25.4.linux-amd64.tar.gz'
    urllib.request.urlretrieve('https://go.dev/dl/go1.25.4.linux-amd64.tar.gz',tgz)
    go_root = WORK/'go'
    if go_root.exists(): shutil.rmtree(go_root)
    with tarfile.open(tgz,'r:gz') as tf: tf.extractall(WORK)
    os.environ['PATH'] = str(WORK/'go/bin')+os.pathsep+os.environ['PATH']
assert go_version() >= (1,25), 'Go 1.25+ required'
run(['go','run','./cmd/adapterbuild','lora','--intent','B2B_IDENTITY_ENRICHMENT','--class','hybrid','--package',str(PACKAGE),'--foundation',str(FOUNDATION/'foundation.json'),'--out',str(OUT)],cwd=KIT)
print('Package bytes:',PACKAGE.stat().st_size)


In [ ]:
# Collect reproducible outputs. This does not host, bond, or register the adapter.
ARTIFACTS = WORK/'tallymark-b2b-artifacts'
ARTIFACTS.mkdir(exist_ok=True)
for src in [PACKAGE,RESULT_PATH,OUT/'B2B_IDENTITY_ENRICHMENT.safetensors.manifest.json',FOUNDATION/'foundation.json']:
    if src.exists(): shutil.copy2(src,ARTIFACTS/src.name)
print('Artifacts:',[p.name for p in ARTIFACTS.iterdir()])
print('Download from Kaggle Output. No wallet or on-chain action is performed.')


## After this notebook

The local model gate and `adapterbuild` are not protocol registration. The next milestone is to verify the required B2B evaluator acceptance criteria, host the package and runnable executor, and complete the protocol's verification process. Do not bond or register until those requirements are confirmed.